In [ ]:
import pandas as pd 

df1 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU200.csv', encoding='cp949')
df2 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU400.csv', encoding='cp949')
df3 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU530.csv', encoding='cp949')
df4 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU300.csv', encoding='cp949')

edu200 = df1.copy()
edu400 = df2.copy()
edu530 = df3.copy()
edu300 = df4.copy()

In [ ]:
# 모든 행과 열 보기.
pd.set_option('display.max_rows', None)  # 모든 행 보기
pd.set_option('display.max_columns', None)  # 모든 열 보기

In [ ]:
import pandas as pd

# 1) 세부 호흡기 질환 매핑 딕셔너리(접두어 매칭)
resp_disease_map = {
    "감기": ["J00","J01","J02","J03","J04","J05","J06"],
    "인플루엔자": ["J09", "J10", "J11"],
    "결핵": ["A15", "A16","A19"],
    "만성폐쇠질환(COPD)": ["J431", "J432","J438","J439","J40","J41","J42","J43","J44","J47"],
    "천식": ["J45","J46"],
    "폐렴": ["A420","A430","B583","B664",
           "J120","J121","J122","J128","J1280","J1288","J129",
            "J13","J14","J150","J151","J152","J153","J154","J155",
            "J156","J157","J158","J159","J160","J168","J170","J171",
            "J173","J178","J180","J181","J188","J189",
            "J690","J691","J698","J853"],
    "기흉": ["J93", "S270", "P251"]
}

def extract_respiratory_and_categories(
    df: pd.DataFrame,
    main_col: str = "주상병코드",
    sub_col: str = "부상병코드",
    disease_map: dict = resp_disease_map,
    include_tb_in_respiratory: bool = False,   # 결핵(A코드)을 전체 호흡기에 포함할지 옵션
    include_nonJ_for_pneumothorax: bool = False  # 기흉 S/P코드 포함 여부 옵션
):
    """
    1) 전체 호흡기질환 환자(주상/부상 중 하나라도 J로 시작) 추출
    2) 그 안에서 매핑 리스트(prefix)로 각 세부 질환 데이터프레임 반환
       - 옵션으로 결핵(A15,A16,A19)이나 기흉의 비-J코드(S270,P251)를 전체 호흡기에 포함할 수 있음
    """
    df = df.copy()

    # 코드 문자열 정규화
    for c in (main_col, sub_col):
        df[c] = df[c].astype(str).str.upper().str.strip()

    # (기본) 전체 호흡기질환: J로 시작하는 코드가 주/부상병 중 하나라도 있으면 포함
    resp_mask = df[main_col].str.startswith("J", na=False) | df[sub_col].str.startswith("J", na=False)
    
    # 결핵(A코드)을 전체 호흡기에 포함하고 싶다면 추가
    if include_tb_in_respiratory and "결핵" in disease_map:
        tb_prefixes = tuple(disease_map["결핵"])
        tb_mask = df[main_col].str.startswith(tb_prefixes, na=False) | df[sub_col].str.startswith(tb_prefixes, na=False)
        resp_mask = resp_mask | tb_mask

    # 기흉의 비-J코드(S270, P251)까지 포함하고 싶다면 추가
    if include_nonJ_for_pneumothorax and "기흉" in disease_map:
        pneumo_extra = [p for p in disease_map["기흉"] if not p.startswith("J")]
        if pneumo_extra:
            px = tuple(pneumo_extra)
            px_mask = df[main_col].str.startswith(px, na=False) | df[sub_col].str.startswith(px, na=False)
            resp_mask = resp_mask | px_mask

    respiratory_df = df[resp_mask].copy()

    # 세부 질환별 부분집합 생성 (prefix 매칭)
    category_dfs = {}
    for name, prefixes in disease_map.items():
        pref = tuple(prefixes)
        m = respiratory_df[main_col].str.startswith(pref, na=False) | respiratory_df[sub_col].str.startswith(pref, na=False)
        category_dfs[name] = respiratory_df[m].copy()

    return respiratory_df, category_dfs

In [ ]:
# -------------------- 사용 예시 --------------------
# 전체 호흡기질환(J*) + (필요시) 결핵/기흉 비-J코드 포함 여부 선택
respiratory_df, cat_dfs = extract_respiratory_and_categories(
    edu200,
    main_col="주상병코드",
    sub_col="부상병코드",
    disease_map=resp_disease_map,
    include_tb_in_respiratory=True,          # 결핵을 전체 호흡기에 포함하려면 True
    include_nonJ_for_pneumothorax=True       # 기흉의 S/P 코드까지 포함하려면 True
)

# 각 세부 질환 데이터셋 접근
influenza_df = cat_dfs["인플루엔자"]
cold_df      = cat_dfs["감기"]
tb_df        = cat_dfs["결핵"]
asthma_df    = cat_dfs['천식']
copd_df      = cat_dfs["만성폐쇠질환(COPD)"]
pneumonia_df = cat_dfs["폐렴"]
pneumo_df    = cat_dfs["기흉"]

# 간단 요약
print("전체 호흡기질환 환자 수:", respiratory_df.shape[0])
for k, v in cat_dfs.items():
    print(f"{k} 환자 수:", v.shape[0])

In [ ]:
# 전체 환자 수
total = pneumonia_df.shape[0]

# 사망자 수 (진료결과 == 4)
deaths = (pneumonia_df['진료결과'] == 4).sum()

# 사망률
mortality_rate = deaths / total

print(f"전체 환자 수: {total}")
print(f"사망자 수: {deaths}")
print(f"사망률: {mortality_rate:.2%}")  # 퍼센트 표시

In [ ]:
# 전체 환자 수
total = respiratory_df.shape[0]

# 사망자 수 (진료결과 == 4)
deaths = (respiratory_df['진료결과'] == 4).sum()

# 사망률
mortality_rate = deaths / total

print(f"전체 환자 수: {total}")
print(f"사망자 수: {deaths}")
print(f"사망률: {mortality_rate:.2%}")  # 퍼센트 표시

In [ ]:
6 / 9 * 100 

In [ ]:
display(pneumonia_df[['주상병코드', '부상병코드']].value_counts())

In [ ]:
pneumonia_df.columns

In [ ]:
pneumonia_df['연령'].value_counts()

In [ ]:
import pandas as pd

# 변환 대상 리스트
dfs = {
    "인플루엔자": influenza_df,
    "감기": cold_df,
    "결핵": tb_df,
    "천식": asthma_df,
    "만성폐쇠질환(COPD)": copd_df,
    "폐렴": pneumonia_df,
    "기흉": pneumo_df
}

# 변환 적용
for name, df in dfs.items():
    dfs[name] = df.apply(pd.to_numeric, errors='ignore')

# 다시 변수로 할당
influenza_df = dfs["인플루엔자"]
cold_df      = dfs["감기"]
tb_df        = dfs["결핵"]
asthma_df    = dfs["천식"]
copd_df      = dfs["만성폐쇠질환(COPD)"]
pneumonia_df = dfs["폐렴"]
pneumo_df    = dfs["기흉"]

# 확인
print(influenza_df.dtypes.head())
print(cold_df.dtypes.head())

In [ ]:
pneumonia_df['연령'].value_counts()

In [ ]:
import pandas as pd

# 연령을 10단위로 묶어서 '연령대'라는 새 컬럼 생성
pneumonia_df['연령대'] = (pneumonia_df['연령'] // 10 * 10).astype(int).astype(str) + "대"

# 결과 확인
print(pneumonia_df[['연령', '연령대']].head())

# 각 연령대별 개수 확인
print(pneumonia_df['연령대'].value_counts().sort_index())

In [ ]:
# 연령대 + 주상병코드 + 부상병코드 조합별 count
age_code_counts = pneumonia_df.groupby(['연령대', '주상병코드', '부상병코드']).size().reset_index(name='count')

display(age_code_counts.head)   # 일부 확인

In [ ]:
# # 부상병명이 '없음'인 경우 주상병코드 목록 추출
# codes_with_no_sub = df_counts.loc[df_counts['부상병명'] == '없음', '부상병코드'].unique()

# print(codes_with_no_sub)

In [ ]:
# 1) 코드 → 병명 매핑 (질문에서 주신 내용 그대로 반영)
code2name = {
    'H663':'만성 화농성 중이염 NOS',
    'C340':'기관분기부의 악성 신생물',
    'N390':'부위가 명시되지 않은 요로감염',
    'N185':'말기신장병 NOS',
    'J128':'기타 바이러스폐렴',
    'C823':'소포성 림프종 등급 IIIa',
    'S141':'경부척수의 기타 및 상세불명의 손상',
    'J304':'상세불명의 알레르기비염',
    'J852':'폐의 농양 NOS',
    'R042':'출혈을 동반한 기침',
    'N183':'만성 신장병(3기)',
    'C220':'간암의 악성 신생물',
    'E079':'상세불명의 갑상선의 장애',
    'D126':'상세불명의 결장의 양성 신생물',
    'C569':'난소의 악성 신생물, 상세불명 쪽',
    'C833':'미만성 대B-세포림프종',
    'K703':'알코올성 간경변증',
    'M329':'상세불명의 전신홍반루푸스',
    'R91':'폐의 진단영상검사상 이상소견',
    'Z038':'기타 의심되는 질환 및 병태의 관찰, 배제된',
    'R509':'상세불명의 열',
    'D509':'상세불명의 철결핍빈혈',
    'A099':'상세불명 기원의 위장염 및 결장염',
    'I269':'폐색전증 NOS',
    'I109':'기타 및 상세불명의 원발성 고혈압',
    'K650':'급성 복막염',
    'R113':'구토를 동반한 구역',
    'NAN': '없음',
    'K210':'역류식도염',
    'H660':'급성 화농성 중이염',
    'B349':'상세불명의 바이러스감염',
    'J939':'상세불명의 기흉',
    'F130':'진정제 또는 수면제 사용에 의한 급성 중독',
    'K769':'상세불명의 간질환',
    'J128':'기타 바이러스폐렴',
    'D143':'기관지 및 폐의 양성 신생물',
    'K297':'상세불명의 위염',
    'B953':'다른 장에서 분류된 질환의 원인으로서의 폐렴연쇄알균',
    'J410':'단순 만성 기관지염',
    'R05':'기침',
    'R509':'발열 NOS',
    'N400':'합병증을 동반하지 않은 전립선증식증',
    'J304':'상세불명의 알레르기비염',
    'N10':'급성 신우염',
    'Z961':'안구내렌즈의 존재',
    'A090':'감염성 기원의 기타 및 상세불명의 위장염 및 결장염',
    'M791':'근통',
    'R068':'무호흡 NOS',
    'E785':'상세불명의 고지질혈증',
    'R51':'두통',
    'M792':'상세불명의 신경통 및 신경염',
    'U099':'상세불명의 코로나-19 이후 병태',
    'J448':'만성 폐색성 기관지염 NOS',
    'J418':'단순 및 점액화농성 혼합형 만성 기관지염',
    'E041':'갑상선 (낭성) 결절 NOS',
    'Z031':'의심되는 악성 신생물에 대한 관찰, 배제된',
    'Z098':' 기타 병태에 대한 기타 치료후 추적검사',
    'S335':'요추의 염좌 및 긴장',
    'L500':'알레르기성 두드러기',
    'D529':'엽산결핍빈혈 NOS',
    'G819':'상세불명의 편마비',
    'K291':'기타 급성 위염',
    'L899':'상세불명의 욕창궤양 및 압박부위',
    'M329':'상세불명의 전신홍반루푸스',
    'U129':'없음',              # 없음
    'K219':'식역(食逆)',
    'J029': '상세불명의 급성 인두염',
    'R074': '상세불명의 흉통',
    'J40':' 급성인지 만성인지 명시되지 않은 기관지염',
    'J690':'음식 또는 구토물에 의한 폐렴',
    'J189':'상세불명의 폐렴',
    'J188':'상세불명 병원체의 기타 폐렴',
    'J128':'기타바이러스 폐렴',
    'J159':'상세불명의 세균폐렴',
    'J961':'만성 호흡부전',
    'J180':'상세불명의 기관지 폐렴',
    'U071':'바이러스가 확인된 코로나바이러스 질환 2019 [바이러스가 확인된 코로나-19]',
    'J189':'J09–J18 인플루엔자 및 폐렴',
    'C509':'유방암 (C50 유방의 악성 신생물)',
    'I603':'대사증후군 (I60–I69 뇌혈관질환 관련 상병)',
    'C349':'C34 폐암',
    'J209':'아데노바이러스 폐렴',
    'G811':'강직성 편마비',
    'J984':'호흡계통의 질환',
    'J449':'기타 만성 폐쇄성 폐질환',
    'J380':'호흡계통의 질환',
    'J690':'호흡계통의 질환',
    'Z115':'기타 바이러스질환에 대한 특수선별검사',
    'C343':'기관지 및 폐의 악성 신생물(C34)',
    'J938':'호흡계통의 질환',
    'J849':'호흡계통의 질환',
    'A192':'A19 좁쌀결핵',
    'J841':'호흡계통의 질환',
    'A153':'A15 세균학적/조직학적으로 확인된 호흡기결핵',
    'J159':'J15 달리 분류되지 않은 세균성 폐렴',
    'R572':'패혈성 쇼크',
    'J111':'J11 바이러스가 확인되지 않은 인플루엔자',
    'J459':'J45 천식',
    'J960':'호흡계통의 질환',
    'E784':'E78 이상지질혈증',
    'J931':'기흉',
    'G473':'수면무호흡',
    'C719':'뇌의 악성 신생물',
    'M321':'전신홍반루푸스',
    'N189':'만성신부전증',
    'J328':'만성 부비동염',
    'G711':'근긴장장애',
    'K801':'담석증(기타 담낭염 동반한 담낭 결석)',
    'J440':'기타 만성 폐쇄성 폐질환',
    'J383':'성대의 기타 질환',
    'I330':'급성 및 아급성 감염성 심내막염',
    'J324':'만성 부비동염',
    'D469':'상세불명의 골수형성이상증후군',
    'K760':'달리 분류되지 않은 지방(변화성)간(비알코올성 지방간질환)',
    'C884':'기관지 관련 림프모양조직의 림프종',
    'M726':'괴사성 근막염',
    'K720':'급성 및 아급성 간부전',
    'C341':'기관지 및 폐의 악성 신생물',
    'J90':'J90–J94 흉막의 기타 질환',
    'J348':'코 및 비동의 기타 명시된 장애',
    'J341':'코 및 비동의 낭 및 점액류',
    'K633':'장의 궤양',
    'J820':'호산구성 기관지염',
    'J869':'누공이 없는 농흉',
    'J441':'기타 만성 폐쇄성 폐질환',
    'I509':'심부전',
    'K802':'담석증(담낭염이 없는 담낭 결석)',
    'J47':'J40–J47 만성 하부호흡기질환',
    'J46':'천식지속상태',
    'M519':'요추 추간판탈출증(허리 디스크)',
    'C310':'부비동의 악성 신생물',
    'A162':'(중분류 A15–A19) 결핵',
    'J188':'인플루엔자 및 폐렴(J09–J18)',
    'J320':'만성 부비동염',
    'C257':'췌장의 악성 신생물',
    'D140':'중이, 비강 및 부비동의 양성 신생물',
    'J171':'달리 분류된 질환에서의 폐렴',
    'N12':'급성/만성으로 명시되지 않은 세뇨관-간질신장염',
    'N133':'폐색성 및 역류성 요로병증',
    'A199':'좁쌀결핵',
    'S525':'아래팔의 골절',
    'F009':'급성 비인두염[감기]',
    'J303':'혈관운동성 및 알레르기성 비염',
    'I352':'기능부전을 동반한 대동맥판협착',
    'S224':'늑골·흉골·흉추의 골절',
    'J36':' 편도주위농양',
    'A150':'세균학적/조직학적으로 확인된 호흡기결핵',
    'J018':'급성 부비동염',
    'J158':'기타 세균 폐렴',
    'H661':'만성 이관고실성 질환',
    'H603':'기타 감염성 외이도염'
}

# 2) 당신이 보여준 표(DataFrame)를 예시로 df_counts 라고 가정
#    (변수명이 다르면 아래 df_counts를 당신의 변수명으로 바꿔서 실행)
df_counts =  pneumonia_df # ← 실제 변수명으로 교체하세요

# 3) 코드 정규화(공백 제거/대문자) 후 매핑해 새 컬럼 추가
df_counts = df_counts.copy()
for col, newcol in [('주상병코드', '주상병명'), ('부상병코드', '부상병명')]:
    df_counts[newcol] = (
        df_counts[col]
        .astype(str).str.strip().str.upper()
        .map(code2name)
        .fillna('없음')
    )

# 4) 결과 확인 (원래 순서 유지)
display(df_counts.head(20))

In [ ]:
df_counts.to_csv("/Users/qwergs/Desktop/df_counts.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_counts['주상병명'].value_counts()

In [ ]:
df_counts['요양기관종별'].value_counts()

In [ ]:
age_institution = (
    df_counts
    .groupby(['연령대', '요양기관종별'])
    .size()
    .reset_index(name='count')
    .sort_values(['연령대', 'count'], ascending=[True, False])
)

print(age_institution.head(20))

In [ ]:
age_institution = (
    df_counts
    .groupby(['연령대', '요양기관종별', '주상병명', '부상병명'])   # 주상병명 추가
    .size()
    .reset_index(name='count')
    .sort_values(['연령대', 'count'], ascending=[True, False])
)

print(age_institution)

In [ ]:
df_counts.info()